# 6. 심화 - 프로그래머가 만나는 것들

알고리즘 교재에는 잘 안 나오지만, 코드를 실제로 굴리기 시작하면 **거의 반드시** 만나는 네 가지다.
전부 "알고리즘은 맞는데 답이 틀린다" 부류의 문제다.

1. **수의 표현과 함정** - 컴퓨터는 0.1을 정확히 저장하지 못한다
2. **참조/복사/해시** - 변수는 값이 아니라 참조다
3. **문자열과 인코딩** - 글자와 바이트는 다르다
4. **지연평가와 정렬** - 다 만들지 않고 흘려보내기, 원하는 기준으로 줄 세우기

## 푸는 방법
- **출력 예측**은 반드시 실행 전에 답을 먼저 적는다. 틀려야 남는다.
- **[직접 풀기]** -> **[정답]** 순서.
- **[실험]** 칸은 답을 맞히는 게 아니라 **직접 돌려보고 관찰**하는 칸이다.

---
# 1부. 수의 표현과 함정

---
## 1.1 유명한 첫 충격

**(1-1) 출력 예측** 셋 다 예측하라.

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(0.1 + 0.2 - 0.3)

**정답:**
```
0.30000000000000004
False
5.551115123125783e-17
```
버그가 아니다. 파이썬만 그런 것도 아니다. C, Java, JavaScript, 엑셀 전부 같다.
**IEEE 754 부동소수점**을 쓰는 모든 환경의 공통 성질이다.

### 왜 그런가

우리가 10진법으로 `1/3`을 정확히 못 쓰는 것과 같다. `0.3333...`은 끝나지 않는다.
컴퓨터는 **2진법**을 쓰는데, `0.1`은 2진법에서 끝나지 않는 소수다.

```
0.1 (10진) = 0.0001100110011001100... (2진, 무한 반복)
```
저장 공간은 유한하므로(64비트) 어딘가에서 **잘린다**. 그래서 실제로 저장된 값은 0.1이 아니다.

**(1-2) 실험** 실제로 저장된 값을 자릿수를 늘려서 본다.

In [ ]:
print("%.20f" % 0.1)
print("%.20f" % 0.2)
print("%.20f" % 0.3)

from decimal import Decimal
print(Decimal(0.1))     # 0.1로 저장된 진짜 값 전부

출력:
```
0.10000000000000000555
0.20000000000000001110
0.29999999999999998890
0.1000000000000000055511151231257827021181583404541015625
```
`0.1`은 실제로 0.1보다 **아주 조금 큰 값**으로 저장돼 있다.
0.1 + 0.2를 하면 그 오차가 합쳐져서 0.3(약간 작게 저장된 값)과 달라진다.

`print(0.1)`이 `0.1`로 보이는 건 파이썬이 **보기 좋게 반올림해서 보여주기** 때문이다.
저장된 값과 보이는 값이 다르다.

---
## 1.2 그래서 실수는 `==`로 비교하지 않는다

**(1-3) 출력 예측** 0.1을 열 번 더하면 1.0인가?

In [ ]:
s = 0.0
for _ in range(10):
    s = s + 0.1
print(s)
print(s == 1.0)
print(s < 1.0)

**정답:** `0.9999999999999999` / `False` / `True`

오차가 **누적**된다. 10번이면 이 정도지만, 10만 번 더하면 눈에 띄게 벌어진다.

**실무 사고 사례:** 걸프전 당시 패트리엇 미사일 시스템이 0.1초 단위로 시간을 누적했는데,
100시간 운용 후 오차가 0.34초까지 벌어져 요격에 실패했다. 같은 원리다.

**(1-4) 출력 예측 - 덧셈 순서가 결과를 바꾼다**

In [ ]:
print(0.1 + 0.2 + 0.3)
print(0.3 + 0.2 + 0.1)
print((0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3))

**정답:** `0.6000000000000001` / `0.6` / `False`

수학에서 덧셈은 결합법칙이 성립하지만 **부동소수점 덧셈은 성립하지 않는다.**
큰 수와 작은 수를 섞어 더할 때는 **작은 것부터** 더하는 게 오차가 적다.

**(1-5) 실수를 비교하는 세 가지 방법**

In [ ]:
import math

a = 0.1 + 0.2
b = 0.3

# (1) 오차 허용 범위(epsilon)를 두고 비교
print(abs(a - b) < 1e-9)

# (2) math.isclose (파이썬 3.5+, 상대오차까지 고려해줌)
print(math.isclose(a, b))

# (3) 애초에 실수를 쓰지 않는다 - 정수로 바꿔서 계산
print(1 + 2 == 3)          # 0.1 단위를 1 단위로

**정답:** 전부 `True`

**(3)이 가장 확실한 방법이다.** 돈 계산은 원 단위 정수로, 시간은 밀리초 정수로 다룬다.
금융 시스템에서 `float`로 금액을 다루면 사고가 난다. 파이썬에는 `decimal.Decimal`도 있다.

**(1-6) 실험** `Decimal`로 하면 어떻게 되나.

In [ ]:
from decimal import Decimal

print(Decimal("0.1") + Decimal("0.2"))
print(Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))

# 주의: 문자열로 넣어야 한다. float를 넣으면 이미 오차가 들어간 값이 들어온다
print(Decimal(0.1) + Decimal(0.2))

**정답:** `0.3` / `True` / `0.3000000000000000166533453694`

`Decimal("0.1")`은 문자열을 10진수 그대로 해석한다. 대신 느리다.

---
## 1.3 큰 수에서 벌어지는 일

파이썬 `int`는 크기 제한이 **없다**(메모리가 허락하는 한). 하지만 `float`는 다르다.

**(1-7) 출력 예측**

In [ ]:
print(2 ** 100)             # 정수
print(1e16 + 1 == 1e16)     # 실수
print(1e16 + 2 == 1e16)

**정답:**
```
1267650600228229401496703205376
True
False
```
`float`는 유효숫자가 약 15~16자리다. `1e16` 근처에서는 **1을 더해도 값이 안 바뀐다.**
표현 가능한 값들 사이의 간격이 이미 1보다 커졌기 때문이다.

파이썬의 `int`가 무제한인 건 특이한 것이다. **C/C++/Java에서는 `int`가 32비트, `long`이 64비트**로
`2**31`, `2**63`을 넘으면 **오버플로우**가 나서 음수로 뒤집힌다.
경쟁 프로그래밍에서 "int로 하면 틀리고 long long으로 해야 맞는" 문제가 이것이다.

---
## 1.4 음수 나눗셈과 나머지

**(1-8) 출력 예측** 여기가 언어마다 다른 지점이다.

In [ ]:
print(7 // 2, 7 % 2)
print(-7 // 2, -7 % 2)
print(7 // -2, 7 % -2)
print(int(-3.7))
print(divmod(-7, 2))

**정답:**
```
3 1
-4 1
-4 -1
-3
(-4, 1)
```
- 파이썬의 `//`는 **버림(floor)**. `-3.5`를 버리면 `-3`이 아니라 **`-4`** 다(더 작은 쪽).
- 파이썬의 `%` 결과 부호는 **오른쪽 피연산자를 따른다.** `-7 % 2`는 **`1`**(양수).
- 반면 `int(-3.7)`은 **0 쪽으로 자른다** -> `-3`. `//`와 다르다.

**C/C++/Java는 `-7 / 2 == -3`, `-7 % 2 == -1`이다.** 파이썬과 반대다.
다른 언어 코드를 옮겨올 때 음수가 섞이면 여기서 조용히 틀린다.

파이썬 방식의 장점: `arr[i % len(arr)]`가 `i`가 음수여도 항상 유효한 인덱스가 된다(원형 배열).

**(1-9) 실험** 원형 배열에서 확인.

In [ ]:
arr = ['a', 'b', 'c', 'd']
for i in range(-4, 5):
    print(i, "->", arr[i % len(arr)])

---
## 1.5 round의 함정

**(1-10) 출력 예측** 우리가 배운 '반올림'과 다르다.

In [ ]:
print(round(0.5), round(1.5), round(2.5), round(3.5))
print(round(2.675, 2))

**정답:** `0 2 2 4` / `2.67`

두 가지가 겹쳐 있다.

**(1) 은행가 반올림(banker's rounding).** 파이썬 `round`는 정확히 `.5`일 때
**가까운 짝수 쪽**으로 간다. 0.5->0, 1.5->2, 2.5->2, 3.5->4.
항상 올리면 많은 값을 더할 때 합계가 계속 커지는 편향이 생기기 때문이다(통계/회계에서 쓰는 방식).

**(2) `round(2.675, 2)`가 2.68이 아닌 이유**는 1.1절 그대로다.
`2.675`가 실제로는 `2.67499999...`로 저장돼 있어서 내림된다.

돈 계산에서 '무조건 올림'이 필요하면 `math.ceil`이나 `Decimal`의 `quantize`를 쓴다.

---
## 1.6 비트 연산

정수를 2진수로 보고 자릿수 단위로 조작한다.
- `&` AND, `|` OR, `^` XOR, `~` NOT
- `<<` 왼쪽 시프트(2배), `>>` 오른쪽 시프트(반)

**(1-11) 출력 예측**

In [ ]:
a, b = 13, 5      # 1101, 0101

print(bin(a), bin(b))
print(a & b)
print(a | b)
print(a ^ b)
print(a << 1, a >> 1)

**정답:**
```
0b1101 0b101
5     (1101 & 0101 = 0101)
13    (1101 | 0101 = 1101)
8     (1101 ^ 0101 = 1000)
26 6  (한 칸 밀면 2배 / 절반, 소수점은 버림)
```
- `&` 둘 다 1일 때만 1
- `|` 하나라도 1이면 1
- `^` **서로 다를 때만** 1

**(1-12) 실무에서 쓰는 관용구** 각각 무엇을 하는지 맞혀라.

In [ ]:
n = 13

print(n & 1)            # (A)
print(n & 1 == 0)       # (B)
print(n >> 1)           # (C)
print(1 << 4)           # (D)

**정답:** `1` / `False` / `6` / `16`

- (A)(B) `n & 1`은 **홀짝 판정**. 마지막 비트가 1이면 홀수. `n % 2`와 같지만 더 빠르다.
- (C) `n >> 1`은 `n // 2`. 이진 탐색의 중간값 계산에 자주 쓴다.
- (D) `1 << k`는 `2**k`.

**(1-13) XOR의 성질 - 출력 예측**

In [ ]:
print(5 ^ 5)
print(5 ^ 0)
print(5 ^ 3 ^ 5)

**정답:** `0` / `5` / `3`

**같은 값을 두 번 XOR하면 사라진다.** `a ^ b ^ a == b`
순서도 상관없다(교환/결합법칙 성립).

**(1-14) 응용 - 하나만 남는 수** 모든 수가 정확히 두 번씩 나오는데 하나만 한 번 나온다. 그 수를 찾아라. **추가 메모리 없이.**

In [ ]:
# [직접 풀기]
arr = [4, 1, 2, 1, 2, 7, 4]
# 힌트: XOR의 성질을 쓰면 딕셔너리도 정렬도 필요 없다

# 답: 7

In [ ]:
# [정답]
arr = [4, 1, 2, 1, 2, 7, 4]

r = 0
for x in arr:
    r ^= x         # 두 번 나온 수는 서로 상쇄되어 0이 된다
print(r)           # 7

딕셔너리로 세면 메모리가 O(n) 필요하지만, XOR은 변수 하나면 된다.
비트 연산은 **플래그 관리**(권한 비트, 상태 플래그), **집합 표현**(부분집합 순회),
**해시 함수** 등에서 계속 나온다.

**(1-15) 실험 - 플래그** 여러 개의 on/off를 정수 하나에 담는다.

In [ ]:
READ  = 1 << 0   # 1  (001)
WRITE = 1 << 1   # 2  (010)
EXEC  = 1 << 2   # 4  (100)

perm = READ | WRITE          # 읽기+쓰기 권한 부여
print(bin(perm), perm)

print(perm & READ != 0)      # 읽기 권한 있나?
print(perm & EXEC != 0)      # 실행 권한 있나?

perm = perm | EXEC           # 실행 권한 추가
print(perm & EXEC != 0)

perm = perm & ~WRITE         # 쓰기 권한 제거
print(perm & WRITE != 0)

리눅스 파일 권한 `chmod 755`가 바로 이것이다. 7 = 111 = 읽기+쓰기+실행.

---
# 2부. 참조 / 복사 / 해시

`a = b` 가 무슨 일을 하는지 정확히 아는 것. 여기서 나는 버그는 **에러가 안 나서** 제일 찾기 어렵다.

---
## 2.1 변수는 값이 아니라 참조다

파이썬에서 변수는 상자가 아니라 **이름표**다. `b = a`는 값을 복사하는 게 아니라
같은 물건에 이름표를 하나 더 붙이는 것이다.

**(2-1) 출력 예측**

In [ ]:
a = [1, 2, 3]
b = a
b.append(4)

print(a)
print(b)
print(a is b)

**정답:** `[1, 2, 3, 4]` / `[1, 2, 3, 4]` / `True`

`b`를 건드렸는데 `a`가 바뀌었다. 둘은 **같은 리스트**다.
`is`는 "같은 물건인가"(주소 비교), `==`는 "내용이 같은가"다.

**(2-2) 출력 예측 - 숫자는 왜 안 그럴까**

In [ ]:
x = 10
y = x
y = y + 1
print(x, y)

a = [1]
b = a
b = b + [2]     # 새 리스트를 만들어 b에 붙임
print(a, b)

c = [1]
d = c
d += [2]        # 이건 다르다!
print(c, d)

**정답:** `10 11` / `[1] [1, 2]` / `[1, 2] [1, 2]`

- 숫자/문자열/튜플은 **불변(immutable)**. 바꿀 수 없으니 항상 새 값이 만들어진다 -> 안전하다.
- 리스트/딕셔너리/집합은 **가변(mutable)**. 원본이 바뀔 수 있다.
- `b = b + [2]`는 **새 리스트를 만들어** 이름을 옮기지만,
  `d += [2]`는 리스트에서 **원본을 그 자리에서 늘린다**(`extend`와 같다). 결과가 다르다.

**규칙: 가변 객체를 다룰 때만 신경 쓰면 된다.**

---
## 2.2 얕은 복사와 깊은 복사

**(2-3) 출력 예측 - 복사하는 방법들**

In [ ]:
a = [1, 2, 3]

b = a[:]            # 슬라이싱
c = list(a)         # 생성자
d = a.copy()        # 메서드

b.append(9)
print(a, b, c, d)
print(a is b, a == b)

**정답:** `[1, 2, 3] [1, 2, 3, 9] [1, 2, 3] [1, 2, 3]` / `False False`

세 방법 다 **새 리스트**를 만든다. 이제 `b`를 건드려도 `a`는 안전하다.

**(2-4) 출력 예측 - 그런데 이건?** 리스트 안에 리스트가 있으면 얘기가 다르다.

In [ ]:
grid = [[1, 2], [3, 4]]
copied = grid[:]        # 복사했다고 믿는다

copied[0][0] = 99

print(grid)
print(copied)

**정답:** 둘 다 `[[99, 2], [3, 4]]`

`grid[:]`는 **바깥 리스트만** 새로 만든다. 안쪽 리스트 2개는 **여전히 공유**하고 있다.
이것이 **얕은 복사(shallow copy)** 다.

```
grid   -> [ *, * ]
              \  copied -> [ *, * ]   <- 바깥 상자는 두 개, 안쪽 내용물은 하나씩 공유
```

**(2-5) 깊은 복사**

In [ ]:
import copy

grid = [[1, 2], [3, 4]]
deep = copy.deepcopy(grid)

deep[0][0] = 99
print(grid)
print(deep)

**정답:** `[[1, 2], [3, 4]]` / `[[99, 2], [3, 4]]`

`deepcopy`는 **안쪽까지 전부** 새로 만든다. 대신 느리고, 순환 참조가 있어도 알아서 처리한다.

2차원 리스트만 필요하면 `[row[:] for row in grid]`로 충분하다(더 빠르다).

**(2-6) 가장 유명한 함정 - 2차원 배열 만들기** 출력을 예측하라.

In [ ]:
g1 = [[0] * 3] * 3
g1[0][0] = 1
print(g1)

g2 = [[0] * 3 for _ in range(3)]
g2[0][0] = 1
print(g2)

**정답:**
```
[[1, 0, 0], [1, 0, 0], [1, 0, 0]]     <- 세 줄이 전부 바뀐다!
[[1, 0, 0], [0, 0, 0], [0, 0, 0]]     <- 정상
```
`[x] * 3`은 **x를 3번 복사하는 게 아니라 x를 가리키는 참조를 3개 만든다.**
`[[0]*3] * 3`은 **같은 행 하나**를 세 번 가리킨 것이다.

**2차원 배열은 반드시 `for`로 만든다.** DFS/BFS의 방문 배열, 동적계획법 표에서 매번 나오는 실수다.

**(2-7) 함정 - 가변 기본 인자** 이건 예측하기 어렵다.

In [ ]:
def add(x, acc=[]):
    acc.append(x)
    return acc

print(add(1))
print(add(2))
print(add(3))

**정답:** `[1]` / `[1, 2]` / `[1, 2, 3]`

기본값 `[]`는 **함수를 정의할 때 딱 한 번** 만들어져서 계속 재사용된다.
호출할 때마다 새로 만들어지지 않는다.

**올바른 작성법:**
```python
def add(x, acc=None):
    if acc is None:
        acc = []
    acc.append(x)
    return acc
```
기본값 자리에는 **불변 객체만** 둔다(숫자, 문자열, `None`, 튜플).

**(2-8) 출력 예측 - 함수에 리스트를 넘기면** 두 함수의 차이는?

In [ ]:
def f1(x):
    x.append(9)

def f2(x):
    x = x + [9]

def f3(x):
    x = x[:]        # 방어적 복사
    x.append(9)
    return x

a = [1]; f1(a); print(a)
b = [1]; f2(b); print(b)
c = [1]; r = f3(c); print(c, r)

**정답:** `[1, 9]` / `[1]` / `[1] [1, 9]`

- `f1`은 원본을 **바꾼다**(호출한 쪽 리스트가 변한다).
- `f2`는 함수 안에서 새 리스트를 만들고 지역 이름에 붙였을 뿐, 밖에는 영향이 없다.
- `f3`처럼 **먼저 복사한 뒤 작업**하면 안전하다.

**함수가 인자를 바꾸는지 안 바꾸는지는 이름이나 문서로 분명히 해야 한다.**
`sort()`(원본 변경) vs `sorted()`(새 객체) 구분이 이 원칙이다.

---
## 2.3 딕셔너리와 집합은 왜 빠른가

**(2-9) 실험 - 속도 차이를 직접 재본다** 맨 끝에 있는 값을 100번 찾아본다.

In [ ]:
import time

N = 200000
lst = list(range(N))
st  = set(lst)
target = N - 1

t = time.perf_counter()
for _ in range(100):
    target in lst
t_list = time.perf_counter() - t

t = time.perf_counter()
for _ in range(100):
    target in st
t_set = time.perf_counter() - t

print("list:", round(t_list, 6), "초")
print("set :", round(t_set, 6), "초")
print("배율:", round(t_list / t_set), "배")

`list`의 `in`은 **처음부터 끝까지 하나씩 비교**한다 -> O(n).
`set`/`dict`의 `in`은 **해시 테이블**을 써서 위치를 바로 계산한다 -> 평균 O(1).

### 해시 테이블의 원리
1. 값을 `hash()` 함수에 넣어 정수를 얻는다.
2. 그 정수로 저장할 **칸 번호**를 계산한다 (`hash(x) % 칸수`).
3. 찾을 때도 똑같이 계산하면 **한 번에** 그 칸으로 간다.

칸 번호가 겹치는 것을 **충돌(collision)** 이라 하고, 파이썬은 옆 칸을 찾아가는 방식으로 처리한다.
충돌이 심하면 최악 O(n)까지 느려질 수 있지만 실제로는 거의 O(1)이다.

**"뭔가를 찾는 코드에서 리스트를 쓰고 있다면 set/dict를 쓸 수 있는지 먼저 본다."**
알고리즘 문제에서 시간 초과의 절반은 이 이유다.

**(2-10) 출력 예측 - 아무거나 키로 쓸 수는 없다**

In [ ]:
d = {}

d[(1, 2)] = "튜플은 된다"
d["문자열"] = "된다"
d[3] = "숫자도 된다"
print(d)

# d[[1, 2]] = "리스트는?"     # 주석 풀면?

**정답:** 마지막 줄은 **TypeError: unhashable type: 'list'**

**가변 객체는 딕셔너리 키가 될 수 없다.** 이유가 명확하다:
키로 넣은 뒤 리스트 내용이 바뀌면 해시값이 달라져서 **다시는 그 칸을 찾을 수 없게** 된다.

키가 될 수 있는 것: 숫자, 문자열, 튜플(단, 안에 리스트가 없어야 함), `frozenset`
키가 될 수 없는 것: 리스트, 딕셔너리, 집합

**그래서 좌표를 키로 쓸 때 `(x, y)` 튜플을 쓴다.** `[x, y]`는 안 된다.

**(2-11) 응용** 방문 여부를 좌표로 관리해보자.

In [ ]:
visited = set()

visited.add((0, 0))
visited.add((1, 2))
visited.add((0, 0))       # 중복은 무시된다

print(visited)
print((1, 2) in visited)
print((5, 5) in visited)
print(len(visited))

**정답:** `{(0, 0), (1, 2)}` (순서는 다를 수 있음) / `True` / `False` / `2`

DFS/BFS에서 방문 배열 대신 이 방식을 쓰면 좌표 범위를 미리 몰라도 된다.

---
# 3부. 문자열과 인코딩

한국에서 프로그래밍하면 **반드시** 겪는다. 파일을 열었는데 한글이 물음표/네모/이상한 한자로 나오는 그것.

---
## 3.1 글자와 바이트는 다르다

파일과 네트워크는 **바이트(byte)** 만 다룬다. 0~255 사이의 숫자 나열이다.
글자를 바이트로 바꾸는 규칙이 **인코딩(encoding)** 이다.
- 글자 -> 바이트: `encode()`
- 바이트 -> 글자: `decode()`

**(3-1) 출력 예측**

In [ ]:
s = "한글"

print(len(s))
print(len(s.encode("utf-8")))
print(len(s.encode("cp949")))
print(s.encode("utf-8"))
print(s.encode("cp949"))
print(type(s), type(s.encode()))

**정답:**
```
2
6
4
b'\xed\x95\x9c\xea\xb8\x80'
b'\xc7\xd1\xb1\xdb'
<class 'str'> <class 'bytes'>
```
같은 "한글"인데 **글자 수는 2, UTF-8 바이트는 6, CP949 바이트는 4.**

- **UTF-8**: 한글 한 자에 **3바이트**. 세계 표준. 웹/리눅스/맥의 기본.
- **CP949 (=EUC-KR 확장)**: 한글 한 자에 **2바이트**. 옛날 윈도우 한국어판의 기본.

`b'...'` 앞의 `b`는 문자열이 아니라 **바이트열**이라는 표시다.
`\xed`는 16진수 한 바이트(=237)를 뜻한다.

**(3-2) 출력 예측 - 영어와 한글**

In [ ]:
print(len("abc"), len("abc".encode("utf-8")))
print("abc".encode("utf-8"))

# 프랑스어 e-acute (U+00E9). 소스에 직접 쓰지 않고 코드번호로 만든다
acute = chr(0x00E9)
cafe2 = "caf" + acute
print(acute, len(cafe2), len(cafe2.encode("utf-8")))

**정답:** `3 3` / `b'abc'` / `(e에 악센트 붙은 글자) 4 5`

**UTF-8은 영어(ASCII)를 1바이트로 쓴다.** 그래서 영어만 있으면 글자 수 = 바이트 수다.
서유럽 문자는 2바이트, 한글/한자는 3바이트, 이모지는 4바이트.
**UTF-8은 글자마다 바이트 수가 다른 가변 길이 인코딩**이다.

영어권 개발자가 만든 코드가 한국에서 터지는 이유가 이것이다.
그쪽에서는 글자 수와 바이트 수가 늘 같아서 구분할 일이 없다.

---
## 3.2 글자 깨짐(mojibake)의 정체

**깨짐 = 인코딩 A로 만든 바이트를 인코딩 B로 읽은 것.**

**(3-3) 실험 - 깨짐을 직접 만들어본다**

In [ ]:
s = "한글"

# UTF-8로 저장한 것을 CP949로 읽으려 하면
b = s.encode("utf-8")
try:
    print(b.decode("cp949"))
except UnicodeDecodeError as e:
    print("에러:", e)

# 에러를 무시하고 억지로 읽으면 (실제 프로그램에서 흔히 이렇게 보인다)
print(repr(b.decode("cp949", errors="replace")))

# 반대 방향
b2 = s.encode("cp949")
print(repr(b2.decode("utf-8", errors="replace")))

**출력:** (물음표 자리에 보이는 것은 U+FFFD "대체 문자". 폰트에 따라 물음표나 검은 마름모로 보인다)
```
에러: 'cp949' codec can't decode byte 0xed in position 0: illegal multibyte sequence
'?븳湲?'          <- 4글자. 가운데 두 글자가 엉뚱한 한글/한자
'?..?'            <- 3글자로 줄어듦. 읽을 수 있는 글자가 거의 없다
```
- **한자와 이상한 한글이 섞여 나오면** -> UTF-8 파일을 CP949로 읽은 것 (한글 3바이트를 2바이트씩 잘라 읽어서)
- **물음표/네모만 잔뜩이면** -> CP949 파일을 UTF-8로 읽은 것 (2바이트가 UTF-8 규칙에 안 맞아 거의 다 버려짐)

깨진 화면만 보고 어느 방향으로 잘못됐는지 **진단할 수 있다.** 실무에서 유용한 감각이다.

주의: 깨진 뒤에 `errors="replace"`로 읽었다면 원래 바이트가 U+FFFD로 **대체되어 사라진다.**
그 문자열은 되돌릴 수 없다. **바이트 원본을 다시 올바른 인코딩으로 읽어야** 한다.

**(3-4) 실험 - 파일 입출력에서** 실제로 사고가 나는 곳.

In [ ]:
import os
path = "encoding_test.txt"

# CP949로 저장
with open(path, "w", encoding="cp949") as f:
    f.write("한글 테스트")

# 인코딩을 지정하지 않고 읽기 -> 환경 기본값을 쓴다 (여기가 위험)
try:
    with open(path, "r", encoding="utf-8") as f:
        print(f.read())
except UnicodeDecodeError as e:
    print("에러:", e)

# 올바르게 읽기
with open(path, "r", encoding="cp949") as f:
    print(f.read())

os.remove(path)

**규칙: `open()`에는 항상 `encoding=`을 명시한다.**

생략하면 파이썬이 **운영체제 기본 인코딩**을 쓴다.
- 윈도우 한국어판: `cp949`
- 리눅스/맥: `utf-8`

그래서 **"내 컴퓨터에서는 되는데 서버에서 안 되는"** 전형적인 사고가 난다.
지금 새로 만드는 것은 무조건 **UTF-8**을 쓴다. CP949는 옛날 파일을 읽을 때만.

**(3-5) 출력 예측 - 개행 문자**

In [ ]:
s = "a\nb"
print(len(s))
print(repr(s))

w = "a\r\nb"
print(len(w))
print(repr(w))
print(repr(w.splitlines()))
print(repr(w.strip()))

**정답:** `3` / `'a\nb'` / `4` / `'a\r\nb'` / `['a', 'b']` / `'a\r\nb'`

- **윈도우는 줄바꿈이 `\r\n` 2바이트**, 리눅스/맥은 `\n` 1바이트다.
- git이 `warning: LF will be replaced by CRLF` 를 띄우는 이유가 이것이다.
- `\r`이 섞이면 `line == "END"` 비교가 조용히 실패한다(실제 값은 `"END\r"`).
  그래서 파일을 줄 단위로 읽으면 **`.strip()`을 습관적으로 붙인다.**

`repr()`을 쓰면 눈에 안 보이는 문자를 볼 수 있다. 디버깅의 기본 도구다.

---
## 3.3 문자 코드와 문자열 만들기

**(3-6) 출력 예측**

In [ ]:
print(ord('A'), ord('a'), ord('0'))
print(chr(65), chr(97))
print(ord('가'), ord('힣'))
print(chr(ord('A') + 2))

**정답:** `65 97 48` / `A a` / `44032 55203` / `C`

- `ord()` 글자 -> 코드번호, `chr()` 코드번호 -> 글자
- 대문자와 소문자는 **32 차이**(`'a' - 'A' == 32`). 비트로 보면 6번째 비트 하나 차이다.
- 완성형 한글은 `가`(44032)부터 `힣`(55203)까지 **연속**으로 배치돼 있다.
  그래서 `chr(44032 + n)`으로 한글을 순회할 수 있다.

카이사르 암호, 문자 개수 세기(`ord(c) - ord('a')`를 인덱스로) 같은 데서 쓴다.

**(3-7) 실험 - 문자열 이어붙이기의 비용** 세 방식의 시간을 재본다.

In [ ]:
import time

N = 100000

# (1) 앞에 붙이기
t = time.perf_counter()
s1 = ""
for i in range(N):
    s1 = "a" + s1
print("앞에 붙이기 :", round(time.perf_counter() - t, 4), "초")

# (2) 뒤에 붙이기
t = time.perf_counter()
s2 = ""
for i in range(N):
    s2 = s2 + "a"
print("뒤에 붙이기 :", round(time.perf_counter() - t, 4), "초")

# (3) 리스트에 모았다가 join
t = time.perf_counter()
parts = []
for i in range(N):
    parts.append("a")
s3 = "".join(parts)
print("join       :", round(time.perf_counter() - t, 4), "초")

print(s1 == s2 == s3)

**문자열은 불변이다.** `s = "a" + s`는 기존 문자열을 늘리는 게 아니라
**길이 n+1짜리 새 문자열을 통째로 만들고** 옛것을 버린다.
n번 반복하면 총 복사량이 1+2+...+n = **O(n^2)** 이 된다. `join`은 **O(n)**.

(1)이 (3)보다 **10배 안팎으로 느리게** 나온다. N을 2배로 키우면 (1)은 4배 느려지고 (3)은 2배만 느려진다.
직접 N을 바꿔가며 확인해보라. 이것이 O(n^2)와 O(n)의 차이다.

(2)가 생각보다 안 느린 것은 CPython이 **"이 문자열을 나 말고 아무도 안 보고 있으면 제자리에서 늘린다"**
는 최적화를 넣어놨기 때문이다. 조건이 하나만 어긋나도(다른 변수가 같은 문자열을 참조 중이라든가,
PyPy 같은 다른 구현이라든가) 최적화가 꺼지고 O(n^2)로 돌아간다.
**최적화에 기대지 말고 `join`을 쓴다.**

---
# 4부. 지연평가와 정렬

"다 만들어놓고 쓰는 것"과 "필요할 때 하나씩 만드는 것"의 차이.

---
## 4.1 range는 리스트가 아니다

**(4-1) 출력 예측**

In [ ]:
r = range(5)
print(r)
print(type(r))
print(list(r))

import sys
print(sys.getsizeof(range(1000000)))
print(sys.getsizeof(list(range(1000000))))

**정답:**
```
range(0, 5)
<class 'range'>
[0, 1, 2, 3, 4]
48
8000056
```
`range(1000000)`은 백만 개를 만들어두지 않는다. **시작/끝/간격 세 숫자만** 기억하고
필요할 때마다 다음 값을 계산해준다. 그래서 크기가 **48바이트로 고정**이다.
리스트로 만들면 **8MB**다.

이것이 **지연평가(lazy evaluation)** 다. 파이썬 2의 `range`는 진짜 리스트를 만들었고,
파이썬 3에서 이렇게 바뀌었다.

**(4-2) 출력 예측 - 대괄호 하나 차이**

In [ ]:
import sys

lst = [x * x for x in range(100000)]     # 리스트 컴프리헨션
gen = (x * x for x in range(100000))     # 제너레이터 표현식

print(type(lst), type(gen))
print(sys.getsizeof(lst))
print(sys.getsizeof(gen))
print(sum(lst) == sum(gen))

**정답:** `<class 'list'> <class 'generator'>` / 약 `800984` / `104` / `True`

**`[]`를 `()`로 바꾸면 제너레이터**가 된다. 결과는 같은데 메모리는 8000배 차이다.

`sum()`, `max()`, `any()`, `all()`처럼 **한 번 훑고 끝나는** 함수에 넘길 때는
리스트를 만들 필요가 전혀 없다.
```python
total = sum(x*x for x in range(10**7))   # 대괄호 없이 - 메모리 안 씀
```

**(4-3) 함정 - 제너레이터는 한 번만 쓸 수 있다**

In [ ]:
gen = (x for x in range(3))

print(list(gen))
print(list(gen))     # 두 번째는?

z = zip([1, 2], [3, 4])
print(list(z))
print(list(z))

**정답:** `[0, 1, 2]` / `[]` / `[(1, 3), (2, 4)]` / `[]`

**한 번 소진되면 비어버린다.** 되감기가 없다.
`zip`, `map`, `filter`, `enumerate`도 전부 같다(파이썬 3 기준).

두 번 써야 하면 **리스트로 받아둔다**: `pairs = list(zip(a, b))`
"분명히 값이 있었는데 두 번째 반복문에서 아무것도 안 돌아간다"의 원인이 이것이다.

---
## 4.2 yield - 직접 만드는 제너레이터

**(4-4) 출력 예측** 출력 순서에 주의하라.

In [ ]:
def gen():
    print("  A 실행")
    yield 1
    print("  B 실행")
    yield 2
    print("  C 실행")

it = gen()
print("함수를 부른 직후")
print(next(it))
print(next(it))

**정답:**
```
함수를 부른 직후
  A 실행
1
  B 실행
2
```
**`gen()`을 호출해도 함수 본문이 실행되지 않는다.** 제너레이터 객체만 만들어진다.
`next()`를 부를 때마다 **`yield`까지 실행하고 그 자리에서 멈춰 있다가**, 다음 `next()`에서 이어서 간다.

`return`은 함수를 끝내지만 `yield`는 **일시정지**한다. 지역 변수가 그대로 살아있다.

**(4-5) 실험 - 무한 수열도 만들 수 있다**

In [ ]:
def naturals():
    n = 0
    while True:        # 끝나지 않는 루프인데
        yield n
        n += 1

it = naturals()
for _ in range(5):
    print(next(it), end=" ")
print()

# 조건에 맞는 것만 필요한 만큼 꺼내기
def evens():
    for x in naturals():
        if x % 2 == 0:
            yield x

e = evens()
print([next(e) for _ in range(5)])

`while True`인데 멈추지 않는다. **필요한 만큼만 만들기** 때문이다.
리스트로는 애초에 표현할 수 없는 것을 다룰 수 있다.

**실무에서 언제 쓰나:** 10GB짜리 로그 파일을 한 줄씩 처리할 때.
```python
with open("huge.log", encoding="utf-8") as f:
    for line in f:        # 파일 객체 자체가 제너레이터다. 통째로 안 읽는다
        process(line)
```
`f.readlines()`로 읽으면 10GB를 전부 메모리에 올린다.

---
## 4.3 정렬 - key가 전부다

**(4-6) 출력 예측**

In [ ]:
w = ['banana', 'apple', 'Cherry', 'date']

print(sorted(w))
print(sorted(w, key=str.lower))
print(sorted(w, key=len))
print(sorted(w, reverse=True))

**정답:**
```
['Cherry', 'apple', 'banana', 'date']
['apple', 'banana', 'Cherry', 'date']
['date', 'apple', 'banana', 'Cherry']
['date', 'banana', 'apple', 'Cherry']
```
첫 줄에서 `Cherry`가 맨 앞인 이유: 문자열 비교는 **코드번호** 순이고 `'C'`(67) < `'a'`(97)이다.
**대문자가 전부 소문자보다 앞선다.** 사람이 기대하는 순서가 아니다.
`key=str.lower`를 주면 비교할 때만 소문자로 바꿔서 본다(원본은 안 바뀐다).

**`key`는 "무엇을 기준으로 비교할지"를 정하는 함수**다. 각 원소에 이 함수를 적용한 결과로 줄을 세운다.

**(4-7) 출력 예측 - 안정 정렬(stable sort)**

In [ ]:
w = ['banana', 'apple', 'Cherry', 'date']
print(sorted(w, key=len))

**정답:** `['date', 'apple', 'banana', 'Cherry']`

길이는 date(4), apple(5), banana(6), Cherry(6)다.
banana와 Cherry는 **길이가 같다.** 그런데 결과에서 banana가 앞에 있다.
**원본에서 banana가 먼저 있었기 때문**이다.

**파이썬의 정렬은 안정(stable)하다: 같은 값끼리는 원래 순서를 유지한다.**
(Timsort라는 알고리즘을 쓴다. 병합 정렬 기반이라 안정적이다.
 퀵 정렬은 안정적이지 않다 - 이래서 언어마다 정렬 결과가 다를 수 있다.)

이 성질을 이용하면 **여러 번 정렬해서 다중 기준을 만들 수 있다:**
```python
data.sort(key=lambda x: x[0])    # 2순위 기준으로 먼저
data.sort(key=lambda x: x[1])    # 1순위 기준으로 나중에 (앞의 순서가 보존됨)
```

**(4-8) 다중 기준 정렬** 점수 내림차순, 점수가 같으면 이름 오름차순으로 정렬하라.

In [ ]:
# [직접 풀기]
data = [('kim', 90), ('lee', 85), ('park', 90), ('choi', 85)]

# 목표: [('kim', 90), ('park', 90), ('choi', 85), ('lee', 85)]

In [ ]:
# [정답]
data = [('kim', 90), ('lee', 85), ('park', 90), ('choi', 85)]

# 튜플을 key로 주면 앞에서부터 차례로 비교한다
print(sorted(data, key=lambda x: (-x[1], x[0])))

# 문자열은 부호를 못 붙이니, 방향이 섞이면 두 번 정렬한다 (안정 정렬 이용)
d = data[:]
d.sort(key=lambda x: x[0])              # 2순위: 이름 오름차순
d.sort(key=lambda x: x[1], reverse=True) # 1순위: 점수 내림차순
print(d)

**핵심 두 가지:**
- **`key`에 튜플을 주면 앞에서부터 순서대로 비교한다.** `(-점수, 이름)` -> 점수 내림, 이름 오름.
- 숫자는 **`-`를 붙여서** 방향을 뒤집을 수 있지만 **문자열은 안 된다.**
  방향이 섞여 있고 문자열이 끼면 **안정 정렬을 이용해 낮은 우선순위부터 여러 번** 정렬한다.

**(4-9) 실험 - key는 몇 번 호출되나**

In [ ]:
count = 0
def k(x):
    global count
    count += 1
    return len(x)

w = ['aaa', 'b', 'cc', 'dddd', 'e', 'ff', 'ggg']
sorted(w, key=k)
print("key 호출:", count, "번 / 원소:", len(w), "개")

**정답:** 원소 개수와 **똑같이** 7번.

파이썬은 정렬 전에 **key를 각 원소에 딱 한 번씩만 적용**해서 결과를 따로 저장해두고,
그 값으로 비교한다(decorate-sort-undecorate 방식).

그래서 `key`에 무거운 계산을 넣어도 O(n)번만 실행된다.
비교할 때마다 계산하는 방식이었다면 O(n log n)번 불렸을 것이다.

---
## 정리

**1부. 수**
- `0.1 + 0.2 != 0.3`. 실수는 `==`로 비교하지 않는다 -> `math.isclose` 또는 정수화
- 오차는 **누적**되고, 덧셈 **순서**에 따라 결과가 달라진다
- 돈/시간은 정수나 `Decimal`로 다룬다
- 파이썬 `//`는 버림(-4), 다른 언어는 0쪽 자름(-3). `%` 부호도 다르다
- `round(2.5) == 2` (은행가 반올림)
- `n & 1` 홀짝, `n >> 1` 반, `1 << k` = 2^k, `a^b^a == b`

**2부. 참조**
- `b = a`는 복사가 아니다. 가변 객체는 같이 바뀐다
- `a[:]`는 **얕은 복사** - 안쪽 리스트는 공유된다
- `[[0]*3]*3` 금지, `[[0]*3 for _ in range(3)]`
- 기본 인자에 `[]`를 두지 않는다 -> `None`을 쓴다
- 찾는 코드에는 `list`가 아니라 `set`/`dict` (O(n) -> O(1))
- 키는 불변 객체만. 좌표는 `(x, y)` 튜플

**3부. 문자열**
- 글자 수 != 바이트 수. UTF-8 한글 3바이트, CP949 2바이트
- `open()`에는 항상 `encoding=` 명시. 새로 만들면 UTF-8
- 깨진 모양으로 방향을 진단할 수 있다
- 윈도우 개행은 `\r\n`. 줄 읽으면 `.strip()`
- 반복문 안 문자열 `+=` 대신 `join`

**4부. 지연평가와 정렬**
- `range`, `zip`, `map`, 제너레이터는 값을 미리 만들지 않는다
- `[...]` -> `(...)` 하나로 메모리가 사라진다
- 제너레이터는 **한 번만** 쓸 수 있다
- `key`로 비교 기준을 정한다. 튜플 key = 다중 기준
- 파이썬 정렬은 **안정 정렬** - 여러 번 정렬로 다중 기준을 만들 수 있다